# External DQ-CIS Complementarity Evaluation

This notebook evaluates whether the locked ROCC history selection adds retrieval evidence beyond the published [DQ-CIS](https://doi.org/10.1145/3742423) ChatGPT/ColBERTv2 rankings on the complete TopiOCQA development split, comparing `DQ`, `DQ+I`, and `DQ+R` under a fixed B64, Top-100, and RRF10 protocol.

Here `DQ` denotes DQ-CIS, while `DQ+I` and `DQ+R` denote its fusion with the ANCE rankings $L_I^{64}$ and $L_R^{64}$. The selector is $f_{\mathrm{base}}$. Both added routes use the total input budget $B=64$, including the current query and history. RRF10 means $\kappa=10$. Both input rankings and the fused output use $k_{\mathrm{ret}}=100$.

### Evaluation setup

B64, Top-100, RRF10, method order, comparisons, and bootstrap seeds are fixed before evaluation, leaving ROCC history selection as the only difference between `DQ+I` and `DQ+R`.

In [1]:
from __future__ import annotations

import gzip
import hashlib
import io
import json
import os
import re
import sys
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def resolve_project_root(start: Path) -> Path:
    configured = os.environ.get("ROCC_ROOT") or os.environ.get("MASTER_THESIS_PROJECT_ROOT")
    if configured:
        candidate = Path(configured).expanduser().resolve()
        if not (candidate / "experiments" / "rocc").is_dir():
            raise FileNotFoundError(f"Configured ROCC checkout is missing: {candidate}")
        return candidate
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "rocc").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from your ROCC checkout, or set ROCC_ROOT to it. "
        "In Colab, first clone the public repository over HTTPS or authenticate "
        "your own private checkout; no private clone or deploy key is assumed."
    )


PROJECT_ROOT = resolve_project_root(Path.cwd())
os.environ["MASTER_THESIS_PROJECT_ROOT"] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.rocc import (
    add_conversation_columns,
    ensure_external_run_file,
    load_jsonl_rankings,
    load_ranked_run_file,
    load_topiocqa_frame,
    paired_bootstrap_mean_ci,
    reciprocal_rank_fusion,
    resolve_topiocqa_resources,
    retrieval_metrics,
    sha256_file,
    topiocqa_dqcis_qid_map,
    unique_ranked_docids,
)


PROTOCOL = "nb08_dqcis_topiocqa_complementarity"
TOP_K = 100
BUDGET = 64
RRF_K = 10
BOOTSTRAP_REPLICATES = 10_000
METHOD_ORDER = ("DQ", "DQ+I", "DQ+R")
COMPARISON_SPECS = (
    ("DQ+R_minus_DQ+I", "DQ+R", "DQ+I"),
    ("DQ+R_minus_DQ", "DQ+R", "DQ"),
    ("DQ+I_minus_DQ", "DQ+I", "DQ"),
)
BOOTSTRAP_SEED_BASE = 1_400

RESULT_DIR = (
    PROJECT_ROOT
    / "experiments"
    / "results"
    / "08_dqcis_external_ablation"
)
EXTERNAL_CACHE_DIR = Path(
    os.environ.get(
        "ROCC_NB08_EXTERNAL_CACHE",
        str(
            PROJECT_ROOT
            / ".cache"
            / "master_thesis"
            / "08_dqcis_external_ablation"
        ),
    )
).expanduser().resolve()
TOPIOCQA_DATA_DIR = (
    PROJECT_ROOT / "experiments" / "data" / "topiocqa"
)
QRECC_GOLD_PATH = (
    PROJECT_ROOT
    / "experiments"
    / "data"
    / "qrecc"
    / "collection"
    / "qrecc-test.gold.json"
)
QRECC_COLLECTION_MANIFEST_PATH = (
    PROJECT_ROOT
    / "experiments"
    / "data"
    / "qrecc"
    / "collection"
    / "manifest.json"
)
NB07A_DIR = (
    PROJECT_ROOT
    / "experiments"
    / "results"
    / "07a_topiocqa_final"
)
TOPI_QUERY_BUNDLE_PATH = (
    NB07A_DIR / "query_bundle" / "queries.jsonl"
)
TOPI_ANCE_RANKINGS_PATH = (
    NB07A_DIR / "backends" / "ance" / "rankings.jsonl.gz"
)

DQ_CIS_URLS = {
    "topiocqa": (
        "https://media.githubusercontent.com/media/"
        "DQCIS/dqcis/main/result/topioqca_chatgpt.txt"
    ),
    "qrecc": (
        "https://media.githubusercontent.com/media/"
        "DQCIS/dqcis/main/result/qrecc_chatgpt.txt"
    ),
}
DQ_CIS_SHA256 = {
    "topiocqa": (
        "9b5a95399b096deeb10ec65d09dade25"
        "b69560ba975d422da1e5d380f8a6f1ae"
    ),
    "qrecc": (
        "42dd9a3eecfbc73bfd24b85da2149a23"
        "d30729f1c2c1047dc2753892d0056d52"
    ),
}
DQ_CIS_FILENAMES = {
    "topiocqa": "topioqca_chatgpt.txt",
    "qrecc": "qrecc_chatgpt.txt",
}
EXPECTED_POPULATION_SHA256 = (
    "f2a71cab647da97ea16471715df20afb"
    "268c92833e2341b9bc64859e9838b6fe"
)
EXPECTED_QUERY_BUNDLE_SHA256 = (
    "c7358e2430302918d218f41e0d55ffb"
    "8a1991773872aadb39e85252008ed28e7"
)
EXPECTED_ANCE_RANKING_SHA256 = (
    "15ef3efc1f5a3ac685a730463748a230"
    "1cea7cc9b77023efdc7a1ee74b2d2cdf"
)
NB05_PRETRAINED_SHA256 = (
    "a07271abefbc37ef08dba4eb94adb0e1"
    "65f53a652060c069c909a7a1614115c4"
)

RESULT_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 180)


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def write_json(path: Path, value: Mapping[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    temporary.write_text(
        json.dumps(
            value,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
            default=str,
        )
        + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def write_frame(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    frame.to_csv(
        temporary,
        index=False,
        float_format="%.17g",
        lineterminator="\n",
    )
    temporary.replace(path)


def write_gzip_frame(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    frame.to_csv(
        temporary,
        index=False,
        float_format="%.17g",
        lineterminator="\n",
        compression={"method": "gzip", "mtime": 0},
    )
    temporary.replace(path)


def write_gzip_jsonl(
    path: Path,
    rows: Iterable[Mapping[str, Any]],
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(f".{path.name}.tmp")
    with temporary.open("wb") as raw:
        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw,
            mtime=0,
        ) as compressed:
            with io.TextIOWrapper(
                compressed,
                encoding="utf-8",
                newline="\n",
            ) as text:
                for row in rows:
                    text.write(
                        json.dumps(
                            row,
                            ensure_ascii=False,
                            sort_keys=True,
                            separators=(",", ":"),
                            default=str,
                        )
                        + "\n"
                    )
    temporary.replace(path)


def population_sha256(sample_ids: Sequence[str]) -> str:
    return sha256_text("\n".join(sorted(map(str, sample_ids))))


def load_query_views(path: Path) -> pd.DataFrame:
    opener = gzip.open if path.suffix == ".gz" else open
    rows: list[dict[str, Any]] = []
    with opener(path, "rt", encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            if (
                str(row.get("family")) == "rocc"
                and str(row.get("system")) == "pretrained"
                and int(row.get("budget", -1)) == BUDGET
                and str(row.get("view")) in {"I", "R"}
            ):
                rows.append(
                    {
                        "sample_id": str(row["sample_id"]),
                        "query_id": str(row["query_id"]),
                        "query_norm": str(row["query_norm"]),
                        "history_depth": int(row["history_depth"]),
                        "view": str(row["view"]),
                    }
                )
    frame = pd.DataFrame(rows)
    if frame.duplicated(["sample_id", "view"]).any():
        raise RuntimeError(f"Duplicate I/R query rows in {path}.")
    return frame.sort_values(
        ["sample_id", "view"],
        kind="mergesort",
    ).reset_index(drop=True)


def rankings_by_view(
    query_views: pd.DataFrame,
    rankings_by_query: Mapping[str, Sequence[str]],
) -> dict[str, dict[str, list[str]]]:
    result = {"I": {}, "R": {}}
    for row in query_views.itertuples(index=False):
        result[str(row.view)][str(row.sample_id)] = list(
            rankings_by_query[str(row.query_id)]
        )
    return result


def read_first_ranked_query(path: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    first_qid: str | None = None
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            parts = line.strip().split()
            if not parts:
                continue
            if len(parts) != 6:
                raise ValueError(
                    f"Malformed run row at {path}:{line_number}."
                )
            qid, _q0, docid, rank, score, _tag = parts
            if first_qid is None:
                first_qid = str(qid)
            if str(qid) != first_qid:
                break
            rows.append(
                {
                    "query_id": str(qid),
                    "rank": int(rank),
                    "docid": str(docid),
                    "score": float(score),
                }
            )
    if not rows:
        raise RuntimeError(f"No ranking rows found in {path}.")
    return pd.DataFrame(rows).sort_values(
        "rank",
        kind="mergesort",
    ).reset_index(drop=True)


def read_first_json_array_item(path: Path) -> dict[str, Any]:
    decoder = json.JSONDecoder()
    buffer = ""
    with path.open("r", encoding="utf-8") as handle:
        while chunk := handle.read(65_536):
            buffer += chunk
            stripped = buffer.lstrip()
            if not stripped.startswith("["):
                raise ValueError(f"Expected a JSON array in {path}.")
            payload = stripped[1:].lstrip()
            try:
                value, _end = decoder.raw_decode(payload)
                if not isinstance(value, dict):
                    raise TypeError(
                        f"First array item in {path} is not an object."
                    )
                return value
            except json.JSONDecodeError:
                continue
    raise RuntimeError(f"Could not read the first JSON item from {path}.")


def has_passage_suffix(docid: str) -> bool:
    return bool(re.search(r"_p\d+$", str(docid)))


def passage_base(docid: str) -> str:
    return re.sub(r"(?:/)?_p\d+$", "", str(docid))


display(
    pd.DataFrame(
        [
            {
                "protocol": PROTOCOL,
                "dataset": "TopiOCQA dev",
                "budget": BUDGET,
                "top_k": TOP_K,
                "rrf_k": RRF_K,
                "bootstrap_replicates": BOOTSTRAP_REPLICATES,
                "result_dir": str(RESULT_DIR),
            }
        ]
    )
)

,protocol,dataset,budget,top_k,rrf_k,bootstrap_replicates,result_dir
0,nb08_dqcis_topiocqa_complementarity,TopiOCQA dev,64,100,10,10000,/experiments/results/08_dqcis_external_ablation


### Artifact identity

Valid comparisons require the exact frozen population, query bundle, ANCE rankings, selector checkpoint, and published DQ-CIS runs. SHA-256 checks bind every input to its expected artifact and detect replaced or mismatched files. Manifest values and physical files are verified separately so a valid manifest cannot hide altered data. The QReCC run is used only for the identifier-compatibility showcase and never enters the evaluation.

In [2]:
dqcis_topi_path = ensure_external_run_file(
    EXTERNAL_CACHE_DIR / DQ_CIS_FILENAMES["topiocqa"],
    url=DQ_CIS_URLS["topiocqa"],
    expected_sha256=DQ_CIS_SHA256["topiocqa"],
    progress=True,
)
dqcis_qrecc_path = ensure_external_run_file(
    EXTERNAL_CACHE_DIR / DQ_CIS_FILENAMES["qrecc"],
    url=DQ_CIS_URLS["qrecc"],
    expected_sha256=DQ_CIS_SHA256["qrecc"],
    progress=True,
)

topi_population_manifest = json.loads(
    (NB07A_DIR / "protocol" / "population.json").read_text(
        encoding="utf-8"
    )
)
topi_bundle_manifest = json.loads(
    (NB07A_DIR / "query_bundle" / "manifest.json").read_text(
        encoding="utf-8"
    )
)
topi_ance_manifest = json.loads(
    (NB07A_DIR / "backends" / "ance" / "manifest.json").read_text(
        encoding="utf-8"
    )
)

# Bind fresh NB07 artifacts only when explicitly requested, and check their
# manifests against the files and the same upstream pretrained checkpoint.
from experiments.rocc.paths import bind_notebook_lineage
nb05_checkpoint = PROJECT_ROOT / "experiments/results/05_history_selector/full_candidates/collapsed_crf/model.pt"
local_inputs = bind_notebook_lineage(
    RESULT_DIR, "nb08", {
        "bundle": EXPECTED_QUERY_BUNDLE_SHA256,
        "rankings": EXPECTED_ANCE_RANKING_SHA256,
        "checkpoint": NB05_PRETRAINED_SHA256,
    }, {"bundle": TOPI_QUERY_BUNDLE_PATH, "rankings": TOPI_ANCE_RANKINGS_PATH, "checkpoint": nb05_checkpoint},
    manifest_fields=(
        ("bundle", NB07A_DIR / "query_bundle/manifest.json", ("query_bundle_sha256",)),
        ("bundle", NB07A_DIR / "backends/ance/manifest.json", ("query_bundle_sha256",)),
        ("rankings", NB07A_DIR / "backends/ance/manifest.json", ("files", "rankings.jsonl.gz")),
        ("checkpoint", nb05_checkpoint.parent / "manifest.json", ("checkpoint_sha256",)),
        ("checkpoint", NB07A_DIR / "query_bundle/manifest.json", ("checkpoint_sha256", "pretrained")),
    ),
)
EXPECTED_QUERY_BUNDLE_SHA256 = local_inputs["bundle"]
EXPECTED_ANCE_RANKING_SHA256 = local_inputs["rankings"]
NB05_PRETRAINED_SHA256 = local_inputs["checkpoint"]

checks = {
    "topiocqa_population": (
        str(topi_population_manifest["sample_id_sha256"]),
        EXPECTED_POPULATION_SHA256,
    ),
    "topiocqa_query_bundle_manifest": (
        str(topi_bundle_manifest["query_bundle_sha256"]),
        EXPECTED_QUERY_BUNDLE_SHA256,
    ),
    "topiocqa_query_bundle_file": (
        sha256_file(TOPI_QUERY_BUNDLE_PATH),
        EXPECTED_QUERY_BUNDLE_SHA256,
    ),
    "topiocqa_ance_manifest": (
        str(topi_ance_manifest["files"]["rankings.jsonl.gz"]),
        EXPECTED_ANCE_RANKING_SHA256,
    ),
    "topiocqa_ance_file": (
        sha256_file(TOPI_ANCE_RANKINGS_PATH),
        EXPECTED_ANCE_RANKING_SHA256,
    ),
    "topiocqa_pretrained_checkpoint": (
        str(
            topi_bundle_manifest["checkpoint_sha256"][
                "pretrained"
            ]
        ),
        NB05_PRETRAINED_SHA256,
    ),
    "dqcis_topiocqa_chatgpt": (
        sha256_file(dqcis_topi_path),
        DQ_CIS_SHA256["topiocqa"],
    ),
    "dqcis_qrecc_chatgpt_showcase": (
        sha256_file(dqcis_qrecc_path),
        DQ_CIS_SHA256["qrecc"],
    ),
}
for name, (observed, expected) in checks.items():
    if observed != expected:
        raise RuntimeError(
            f"Artifact identity check failed for {name}: "
            f"{observed} != {expected}"
        )

artifact_identity = pd.DataFrame(
    [
        {
            "artifact": name,
            "sha256": observed,
            "valid": observed == expected,
        }
        for name, (observed, expected) in checks.items()
    ]
)
display(artifact_identity)

,artifact,sha256,valid
0,topiocqa_population,f2a71cab647da97ea16471715df20afb268c92833e2341b9bc64859e9838b6fe,True
1,topiocqa_query_bundle_manifest,c7358e2430302918d218f41e0d55ffb8a1991773872aadb39e85252008ed28e7,True
2,topiocqa_query_bundle_file,c7358e2430302918d218f41e0d55ffb8a1991773872aadb39e85252008ed28e7,True
3,topiocqa_ance_manifest,15ef3efc1f5a3ac685a730463748a2301cea7cc9b77023efdc7a1ee74b2d2cdf,True
4,topiocqa_ance_file,15ef3efc1f5a3ac685a730463748a2301cea7cc9b77023efdc7a1ee74b2d2cdf,True
5,topiocqa_pretrained_checkpoint,a07271abefbc37ef08dba4eb94adb0e165f53a652060c069c909a7a1614115c4,True
6,dqcis_topiocqa_chatgpt,9b5a95399b096deeb10ec65d09dade25b69560ba975d422da1e5d380f8a6f1ae,True
7,dqcis_qrecc_chatgpt_showcase,42dd9a3eecfbc73bfd24b85da2149a23d30729f1c2c1047dc2753892d0056d52,True


### QReCC identifier incompatibility

The published DQ-CIS run and the local QReCC evaluation use different retrieval units. Its first ranking already contains 70 page URLs and 30 passage URLs, while the local qrels and index identify individual passages through numeric IDs. Passage URLs can potentially be mapped to local IDs, but page URLs omit the required passage identity. One demonstrated page corresponds to nine distinct passages, so choosing one passage would be arbitrary and expanding the page would change the published ranking.

MRR, recall, and RRF require an exact shared document-ID space. Unmapped pages would create false misses, while guessed mappings would create unsupported matches. The QReCC run therefore cannot support faithful metrics or fusion and remains only an identifier-compatibility showcase.

In [3]:
qrecc_first_ranking = read_first_ranked_query(
    dqcis_qrecc_path
)
page_level_rows = qrecc_first_ranking.loc[
    ~qrecc_first_ranking["docid"].map(has_passage_suffix)
].copy()
passage_level_rows = qrecc_first_ranking.loc[
    qrecc_first_ranking["docid"].map(has_passage_suffix)
].copy()
if page_level_rows.empty:
    raise RuntimeError(
        "The QReCC compatibility premise changed: no page-level IDs "
        "were found in the first published ranking."
    )

passages_by_page: dict[str, set[str]] = {}
for docid in passage_level_rows["docid"].astype(str):
    passages_by_page.setdefault(
        passage_base(docid),
        set(),
    ).add(docid)
ambiguous_page, ambiguous_variants = max(
    passages_by_page.items(),
    key=lambda item: len(item[1]),
)
if len(ambiguous_variants) < 2:
    raise RuntimeError(
        "The QReCC showcase found no page with multiple passage IDs."
    )

local_gold_row = read_first_json_array_item(
    QRECC_GOLD_PATH
)
local_context = local_gold_row["positive_ctxs"][0]
local_numeric_id = str(
    local_gold_row["positive_ctx_passage_ids"][0]
)
local_raw_id = str(local_context["raw_passage_id"])
collection_manifest = json.loads(
    QRECC_COLLECTION_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)
if not has_passage_suffix(local_raw_id):
    raise RuntimeError(
        "The local QReCC gold example is not passage-specific."
    )
if (
    collection_manifest["collection_format"]["id"]
    != "numeric string"
):
    raise RuntimeError(
        "The local QReCC index ID schema changed."
    )

external_page_example = page_level_rows.iloc[0]
external_passage_example = passage_level_rows.loc[
    passage_level_rows["docid"].map(passage_base).eq(
        ambiguous_page
    )
].iloc[0]
qrecc_identifier_examples = pd.DataFrame(
    [
        {
            "source": "Published DQ-CIS",
            "query_id": external_page_example["query_id"],
            "rank": int(external_page_example["rank"]),
            "identifier": external_page_example["docid"],
            "scope": "page",
            "passage_suffix_present": False,
            "usable_as_local_numeric_docid": False,
        },
        {
            "source": "Published DQ-CIS",
            "query_id": external_passage_example["query_id"],
            "rank": int(external_passage_example["rank"]),
            "identifier": external_passage_example["docid"],
            "scope": "passage",
            "passage_suffix_present": True,
            "usable_as_local_numeric_docid": False,
        },
        {
            "source": "Local QReCC qrel",
            "query_id": (
                f"{local_gold_row['Conversation_no']}_"
                f"{local_gold_row['Turn_no']}"
            ),
            "rank": None,
            "identifier": local_raw_id,
            "scope": "passage",
            "passage_suffix_present": True,
            "usable_as_local_numeric_docid": False,
        },
        {
            "source": "Local QReCC index",
            "query_id": (
                f"{local_gold_row['Conversation_no']}_"
                f"{local_gold_row['Turn_no']}"
            ),
            "rank": None,
            "identifier": local_numeric_id,
            "scope": "numeric passage",
            "passage_suffix_present": None,
            "usable_as_local_numeric_docid": True,
        },
    ]
)

qrecc_compatibility = pd.DataFrame(
    [
        {
            "published_query": str(
                qrecc_first_ranking.iloc[0]["query_id"]
            ),
            "published_hits": len(qrecc_first_ranking),
            "page_level_ids_without_passage": len(
                page_level_rows
            ),
            "passage_level_ids": len(
                passage_level_rows
            ),
            "example_page_with_multiple_passages": (
                ambiguous_page
            ),
            "distinct_passages_for_example_page": len(
                ambiguous_variants
            ),
            "complete_exact_crosswalk": False,
            "eligible_for_metrics_or_fusion": False,
        }
    ]
)

display(qrecc_identifier_examples)
display(qrecc_compatibility)

,source,query_id,rank,identifier,scope,passage_suffix_present,usable_as_local_numeric_docid
0,Published DQ-CIS,1_1,1.0,https://www.franciscanhealth.org/health,page,False,False
1,Published DQ-CIS,1_1,6.0,http://link.library.missouri.edu/resource/0frpAMqCUsI/_p6,passage,True,False
2,Local QReCC qrel,1_2,NaN,http://web.archive.org/web/20200524095112id_/https://www.geteducated.com/careers/how-to-become-a-physician-assistant/_p0,passage,True,False
3,Local QReCC index,1_2,NaN,54355562,numeric passage,None,True


,published_query,published_hits,page_level_ids_without_passage,passage_level_ids,example_page_with_multiple_passages,distinct_passages_for_example_page,complete_exact_crosswalk,eligible_for_metrics_or_fusion
0,1_1,100,70,30,http://link.library.missouri.edu/resource/0frpAMqCUsI,9,False,False


The published ranking mixes page-level and passage-level URLs. Rank 1 identifies only a page, while rank 6 identifies a specific passage. The local qrel also uses a passage URL, but the local index represents that passage as the numeric ID `54355562`.

This difference cannot be solved by renaming IDs. The first published query contains 70 page-level and 30 passage-level hits, and one page has nine distinct passage variants. A page-level hit therefore cannot be assigned to one local passage without changing its meaning. Exact metrics and RRF fusion are consequently invalid for the published QReCC run.

### Population and run alignment

Valid paired comparisons require every method to retrieve for the same conversational turn and with the same ranking depth. The frozen TopiOCQA population contains 2,514 queries from 205 conversations, and its hash matches NB07a. Every published DQ-CIS query ID maps one-to-one to a local `sample_id`, with no missing or duplicated queries. All rankings contain 100 unique documents.

Later differences between `DQ`, `DQ+I`, and `DQ+R` therefore cannot result from population loss, incorrect turn alignment, or shorter external rankings. They measure ranking complementarity on identical queries.

In [4]:
topi_resources = resolve_topiocqa_resources(
    TOPIOCQA_DATA_DIR
)
topi_frame = add_conversation_columns(
    load_topiocqa_frame(
        topi_resources,
        splits=("dev",),
    )
).copy()
topi_frame["sample_id"] = topi_frame[
    "sample_id"
].astype(str)
if len(topi_frame) != 2_514:
    raise RuntimeError(
        "TopiOCQA must contain 2,514 development queries."
    )
if topi_frame["sample_id"].duplicated().any():
    raise RuntimeError(
        "TopiOCQA contains duplicate sample IDs."
    )

topi_population = topi_frame[
    ["sample_id", "conv_id", "turn_id"]
].copy()
topi_gold = {
    str(row.sample_id): unique_ranked_docids(
        row.positive_ctx_passage_ids,
        top_k=max(
            1,
            len(row.positive_ctx_passage_ids),
        ),
    )
    for row in topi_frame.itertuples(index=False)
}
observed_population_sha256 = population_sha256(
    topi_population["sample_id"].tolist()
)
if observed_population_sha256 != EXPECTED_POPULATION_SHA256:
    raise RuntimeError(
        "The TopiOCQA population hash changed."
    )

dqcis_external_run = load_ranked_run_file(
    dqcis_topi_path,
    top_k=TOP_K,
)
topi_qid_to_sample = topiocqa_dqcis_qid_map(
    topi_population["sample_id"],
    dqcis_external_run,
)
dqcis_by_sample = {
    sample_id: dqcis_external_run[qid]
    for qid, sample_id in topi_qid_to_sample.items()
}
expected_sample_ids = set(
    topi_population["sample_id"]
)
if set(dqcis_by_sample) != expected_sample_ids:
    raise RuntimeError(
        "The DQ-CIS run does not cover the TopiOCQA population."
    )

qid_mapping = pd.DataFrame(
    [
        {
            "dqcis_qid": qid,
            "sample_id": sample_id,
            "conv_id": int(
                sample_id.split(":")[1]
            ),
            "turn_id": int(
                sample_id.split(":")[2]
            ),
        }
        for qid, sample_id in topi_qid_to_sample.items()
    ]
).sort_values(
    ["conv_id", "turn_id"],
    kind="mergesort",
).reset_index(drop=True)

dq_depths = np.asarray(
    [
        len(docids)
        for docids in dqcis_by_sample.values()
    ],
    dtype=int,
)
population_summary = pd.DataFrame(
    [
        {
            "dataset": "TopiOCQA dev",
            "queries": len(topi_population),
            "conversations": int(
                topi_population["conv_id"].nunique()
            ),
            "sample_id_sha256": (
                observed_population_sha256
            ),
        }
    ]
)
dqcis_depth_summary = pd.DataFrame(
    [
        {
            "queries": len(dq_depths),
            "min_depth": int(dq_depths.min()),
            "mean_depth": float(dq_depths.mean()),
            "max_depth": int(dq_depths.max()),
            "shorter_than_100": int(
                (dq_depths < TOP_K).sum()
            ),
        }
    ]
)

display(population_summary)
display(dqcis_depth_summary.round(4))

,dataset,queries,conversations,sample_id_sha256
0,TopiOCQA dev,2514,205,f2a71cab647da97ea16471715df20afb268c92833e2341b9bc64859e9838b6fe


,queries,min_depth,mean_depth,max_depth,shorter_than_100
0,2514,100,100.0,100,0


All 2,514 queries from 205 conversations match the frozen TopiOCQA population. Every DQ-CIS ranking contains exactly 100 unique hits, with no missing or shorter runs. All comparisons therefore use the same queries and retrieval depth, eliminating coverage and truncation as alternative explanations for performance differences.

### ROCC view alignment and fusion eligibility

DQ-CIS produces semantic rankings through ChatGPT query expansion and ColBERTv2. ROCC therefore contributes its frozen ANCE rankings from NB07a, keeping the ablation within dense semantic retrieval. A BM25 variant would constitute a separate lexical–semantic complementarity experiment.

`DQ`, `I`, and `R` contain 100 results per query and use the same numeric passage IDs, enabling exact document-level RRF fusion. For the 205 queries without history, `I` and `R` are identical. For all remaining queries, `DQ+I` and `DQ+R` differ only in whether IterCQR receives recency-truncated or ROCC-selected history under the same B64 budget. Their performance difference directly measures the retrieval value of ROCC’s selection.

In [5]:
topi_query_views = load_query_views(
    TOPI_QUERY_BUNDLE_PATH
)
if len(topi_query_views) != 2 * len(topi_population):
    raise RuntimeError(
        "The TopiOCQA I/R query bundle is incomplete."
    )

topi_rankings_by_query = load_jsonl_rankings(
    TOPI_ANCE_RANKINGS_PATH,
    query_ids=set(topi_query_views["query_id"]),
    top_k=TOP_K,
)
topi_views = rankings_by_view(
    topi_query_views,
    topi_rankings_by_query,
)

view_rows = []
for view in ("I", "R"):
    if set(topi_views[view]) != expected_sample_ids:
        raise RuntimeError(
            f"The TopiOCQA {view} rankings are incomplete."
        )
    depths = np.asarray(
        [
            len(ranking)
            for ranking in topi_views[view].values()
        ],
        dtype=int,
    )
    view_rows.append(
        {
            "view": view,
            "queries": len(depths),
            "min_depth": int(depths.min()),
            "mean_depth": float(depths.mean()),
            "max_depth": int(depths.max()),
        }
    )

no_history = topi_query_views.loc[
    topi_query_views["history_depth"].eq(0)
]
no_history_queries = no_history.pivot(
    index="sample_id",
    columns="view",
    values="query_norm",
)
if not no_history_queries["I"].eq(
    no_history_queries["R"]
).all():
    raise RuntimeError(
        "I and R differ for a query without history."
    )
for sample_id in no_history_queries.index.astype(str):
    if (
        topi_views["I"][sample_id]
        != topi_views["R"][sample_id]
    ):
        raise RuntimeError(
            "I and R rankings differ for a query without history."
        )

dqcis_numeric = all(
    str(docid).isdigit()
    for ranking in dqcis_by_sample.values()
    for docid in ranking
)
ance_numeric = all(
    str(docid).isdigit()
    for view in ("I", "R")
    for ranking in topi_views[view].values()
    for docid in ranking
)
if not (dqcis_numeric and ance_numeric):
    raise RuntimeError(
        "TopiOCQA DQ-CIS and ANCE do not share the numeric "
        "passage-ID namespace."
    )

view_summary = pd.DataFrame(view_rows)
namespace_summary = pd.DataFrame(
    [
        {
            "dqcis_docids_numeric": dqcis_numeric,
            "ance_docids_numeric": ance_numeric,
            "exact_namespace_compatible": True,
            "no_history_queries_checked": len(
                no_history_queries
            ),
        }
    ]
)
display(view_summary.round(4))
display(namespace_summary)

,view,queries,min_depth,mean_depth,max_depth
0,I,2514,100,100.0,100
1,R,2514,100,100.0,100


,dqcis_docids_numeric,ance_docids_numeric,exact_namespace_compatible,no_history_queries_checked
0,True,True,True,205


`I` and `R` cover all 2,514 queries with exactly 100 ranked passages each. DQ-CIS and ANCE share the same numeric passage-ID namespace, enabling exact document-level fusion. All 205 no-history cases preserve identical `I` and `R` rankings, confirming that observed differences originate from selected conversational history.

### Controlled dense-list fusion

`DQ` preserves the published DQ-CIS Top-100 ranking. `DQ+I` adds the budgeted baseline $I_{64}$ IterCQR→ANCE ranking, while `DQ+R` adds the ROCC-selected IterCQR→ANCE ranking. Both systems fuse exactly two semantic Top-100 lists with unweighted RRF10 and retain 100 unique passages.

In [6]:
runs: dict[str, dict[str, list[str]]] = {
    method: {} for method in METHOD_ORDER
}
run_identity_rows = []

for sample_id in sorted(expected_sample_ids):
    dq_ranking = unique_ranked_docids(
        dqcis_by_sample[sample_id],
        top_k=TOP_K,
    )
    runs["DQ"][sample_id] = dq_ranking
    run_identity_rows.append(
        {
            "sample_id": sample_id,
            "system": "DQ",
            "input_lists": 1,
            "dq_depth": len(dq_ranking),
            "added_depth": 0,
            "input_overlap": 0,
            "output_depth": len(dq_ranking),
        }
    )

    for system, view in (
        ("DQ+I", "I"),
        ("DQ+R", "R"),
    ):
        added = unique_ranked_docids(
            topi_views[view][sample_id],
            top_k=TOP_K,
        )
        fused_hits = reciprocal_rank_fusion(
            [dq_ranking, added],
            rrf_k=RRF_K,
            top_k=TOP_K,
        )
        fused_docids = [
            str(hit["docid"])
            for hit in fused_hits
        ]
        runs[system][sample_id] = fused_docids
        run_identity_rows.append(
            {
                "sample_id": sample_id,
                "system": system,
                "input_lists": 2,
                "dq_depth": len(dq_ranking),
                "added_depth": len(added),
                "input_overlap": len(
                    set(dq_ranking).intersection(added)
                ),
                "output_depth": len(fused_docids),
            }
        )

for system in METHOD_ORDER:
    if set(runs[system]) != expected_sample_ids:
        raise RuntimeError(
            f"The {system} run is incomplete."
        )
    for sample_id, docids in runs[system].items():
        if len(docids) > TOP_K:
            raise RuntimeError(
                f"{system}/{sample_id} exceeds Top-{TOP_K}."
            )
        if len(docids) != len(set(docids)):
            raise RuntimeError(
                f"{system}/{sample_id} contains duplicate documents."
            )

run_identity = pd.DataFrame(run_identity_rows)
run_identity_summary = (
    run_identity.groupby(
        "system",
        observed=True,
    )
    .agg(
        queries=("sample_id", "size"),
        input_lists=("input_lists", "mean"),
        mean_dq_depth=("dq_depth", "mean"),
        mean_added_depth=("added_depth", "mean"),
        mean_input_overlap=("input_overlap", "mean"),
        mean_output_depth=("output_depth", "mean"),
    )
    .reset_index()
)
run_identity_summary["system"] = pd.Categorical(
    run_identity_summary["system"],
    categories=METHOD_ORDER,
    ordered=True,
)
run_identity_summary = run_identity_summary.sort_values(
    "system",
    kind="mergesort",
).reset_index(drop=True)
run_identity_summary["system"] = (
    run_identity_summary["system"].astype(str)
)
display(run_identity_summary.round(4))

,system,queries,input_lists,mean_dq_depth,mean_added_depth,mean_input_overlap,mean_output_depth
0,DQ,2514,1.0,100.0,0.0,0.0000,100.0
1,DQ+I,2514,2.0,100.0,100.0,12.0398,100.0
2,DQ+R,2514,2.0,100.0,100.0,15.6205,100.0


All 2,514 queries use the same ranking depths and output cutoff. `DQ+I` and `DQ+R` each fuse two complete Top-100 lists.

`DQ+R` shares an average of 15.62 passages with DQ-CIS, compared with 12.04 for `DQ+I`. ROCC selection increases agreement by 3.58 passages per query, showing that selected history directs ANCE toward more of the semantic evidence identified by DQ-CIS. Retrieval metrics determine whether this stronger agreement improves relevance.

### Query-level retrieval effectiveness

Each `DQ`, `DQ+I`, and `DQ+R` ranking is scored against the same gold passages for all 2,514 queries.

In [7]:
population_index = topi_population.set_index(
    "sample_id"
)
metric_rows = []

for system in METHOD_ORDER:
    for sample_id in sorted(runs[system]):
        docids = runs[system][sample_id]
        metrics = retrieval_metrics(
            [
                {
                    "docid": docid,
                    "rank": rank,
                }
                for rank, docid in enumerate(
                    docids,
                    start=1,
                )
            ],
            topi_gold[sample_id],
            ks=(3, 10, 100),
        )
        population_row = population_index.loc[
            sample_id
        ]
        metric_rows.append(
            {
                "system": system,
                "sample_id": sample_id,
                "conv_id": int(
                    population_row["conv_id"]
                ),
                "turn_id": int(
                    population_row["turn_id"]
                ),
                "num_hits": len(docids),
                "target_rank": metrics["target_rank"],
                "MRR": float(metrics["MRR"]),
                "nDCG@3": float(metrics["nDCG@3"]),
                "R@10": float(metrics["R@10"]),
                "R@100": float(metrics["R@100"]),
            }
        )

metrics_by_query = pd.DataFrame(metric_rows)
summary = (
    metrics_by_query.groupby(
        "system",
        observed=True,
    )
    .agg(
        n=("sample_id", "size"),
        mean_depth=("num_hits", "mean"),
        MRR=("MRR", "mean"),
        **{
            "nDCG@3": ("nDCG@3", "mean"),
            "R@10": ("R@10", "mean"),
            "R@100": ("R@100", "mean"),
        },
    )
    .reset_index()
)
summary["system"] = pd.Categorical(
    summary["system"],
    categories=METHOD_ORDER,
    ordered=True,
)
summary = summary.sort_values(
    "system",
    kind="mergesort",
).reset_index(drop=True)
summary["system"] = summary["system"].astype(str)
display(summary.round(4))

,system,n,mean_depth,MRR,nDCG@3,R@10,R@100
0,DQ,2514,100.0,0.3310,0.3244,0.5123,0.6834
1,DQ+I,2514,100.0,0.3320,0.3185,0.5732,0.7649
2,DQ+R,2514,100.0,0.3461,0.3287,0.5879,0.7924


Adding the budgeted baseline $I_{64}$ route mainly increases coverage. Compared with `DQ`, `DQ+I` raises $\mathrm{Recall}@10$ by 0.0609 and $\mathrm{Recall}@100$ by 0.0815, while MRR changes by only 0.0010 and nDCG@3 decreases by 0.0058. The added baseline ranking increases recall without a supported MRR gain.

`DQ+R` achieves the strongest result across every metric. Compared with `DQ+I`, ROCC selection adds 0.0141 MRR, 0.0102 nDCG@3, 0.0147 $\mathrm{Recall}@10$, and 0.0274 $\mathrm{Recall}@100$. The selected history gives IterCQR a more focused conversational context, improving both early ranking and evidence coverage. The paired bootstrap evaluates the stability of these gains across queries.

### MRR comparisons

`DQ+R − DQ+I` isolates the effect of replacing recency-truncated history with ROCC-selected history at B64. `DQ+R − DQ` measures the total value added to the published DQ-CIS ranking. `DQ+I − DQ` tests whether adding the budgeted baseline dense route already produces the same benefit.

MRR differences are computed for the same 2,514 queries. The paired query-level bootstrap follows the inference protocol used throughout the preceding ROCC evaluations.

In [8]:
pivot = metrics_by_query.pivot(
    index="sample_id",
    columns="system",
    values="MRR",
).reset_index()
if (
    len(pivot) != len(topi_population)
    or pivot[list(METHOD_ORDER)].isna().any().any()
):
    raise RuntimeError(
        "The paired TopiOCQA MRR table is incomplete."
    )

comparison_rows = []
for comparison_index, (
    comparison,
    left,
    right,
) in enumerate(COMPARISON_SPECS):
    deltas = (
        pivot[left].to_numpy(dtype=float)
        - pivot[right].to_numpy(dtype=float)
    )
    seed = (
        BOOTSTRAP_SEED_BASE
        + comparison_index
    )
    ci_low, ci_high = paired_bootstrap_mean_ci(
        deltas,
        seed=seed,
        replicates=BOOTSTRAP_REPLICATES,
        progress=False,
    )
    comparison_rows.append(
        {
            "comparison": comparison,
            "left": left,
            "right": right,
            "n_queries": len(deltas),
            "left_mrr": float(pivot[left].mean()),
            "right_mrr": float(pivot[right].mean()),
            "delta_mrr": float(deltas.mean()),
            "ci95_low": float(ci_low),
            "ci95_high": float(ci_high),
            "seed": seed,
            "replicates": BOOTSTRAP_REPLICATES,
        }
    )

paired_comparisons = pd.DataFrame(
    comparison_rows
)
display(paired_comparisons.round(4))

,comparison,left,right,n_queries,left_mrr,right_mrr,delta_mrr,ci95_low,ci95_high,seed,replicates
0,DQ+R_minus_DQ+I,DQ+R,DQ+I,2514,0.3461,0.332,0.0141,0.0045,0.0238,1400,10000
1,DQ+R_minus_DQ,DQ+R,DQ,2514,0.3461,0.331,0.0150,0.0019,0.0280,1401,10000
2,DQ+I_minus_DQ,DQ+I,DQ,2514,0.3320,0.331,0.0010,-0.0115,0.0138,1402,10000


`DQ+R` improves MRR over `DQ+I` by 0.0141, equivalent to a 4.2% relative gain. Its 95% confidence interval `[0.0045, 0.0238]` remains above zero.

`DQ+I` improves over `DQ` by only 0.0010, with its confidence interval spanning zero. Adding the $I_{64}$ ANCE list alone provides no supported MRR gain. `DQ+R` exceeds `DQ` by 0.0150 with a positive confidence interval `[0.0019, 0.0280]`.

The robust result is the controlled comparison between `DQ+R` and `DQ+I`. ROCC’s selected history makes the added dense ranking more effective than the same pipeline using recency-truncated history at B64.

### Final manifest

Rankings, query-level metrics, and comparisons are persisted for direct reconstruction. SHA-256 binds all inputs, settings, and outputs. The manifest records a fixed TopiOCQA evaluation without model selection or QReCC metrics.

In [9]:
output_paths = {
    "artifact_identity.csv": RESULT_DIR / "artifact_identity.csv",
    "qrecc_identifier_examples.csv": (
        RESULT_DIR / "qrecc_identifier_examples.csv"
    ),
    "qrecc_compatibility_showcase.csv": (
        RESULT_DIR / "qrecc_compatibility_showcase.csv"
    ),
    "population_summary.csv": (
        RESULT_DIR / "population_summary.csv"
    ),
    "dqcis_depth_summary.csv": (
        RESULT_DIR / "dqcis_depth_summary.csv"
    ),
    "view_summary.csv": RESULT_DIR / "view_summary.csv",
    "namespace_summary.csv": (
        RESULT_DIR / "namespace_summary.csv"
    ),
    "run_identity_summary.csv": (
        RESULT_DIR / "run_identity_summary.csv"
    ),
    "summary.csv": RESULT_DIR / "summary.csv",
    "paired_comparisons.csv": (
        RESULT_DIR / "paired_comparisons.csv"
    ),
    "metrics_by_query.csv.gz": (
        RESULT_DIR / "metrics_by_query.csv.gz"
    ),
    "qid_mapping.csv": RESULT_DIR / "qid_mapping.csv",
    "rankings_top100.jsonl.gz": (
        RESULT_DIR / "rankings_top100.jsonl.gz"
    ),
}

write_frame(
    output_paths["artifact_identity.csv"],
    artifact_identity,
)
write_frame(
    output_paths["qrecc_identifier_examples.csv"],
    qrecc_identifier_examples,
)
write_frame(
    output_paths["qrecc_compatibility_showcase.csv"],
    qrecc_compatibility,
)
write_frame(
    output_paths["population_summary.csv"],
    population_summary,
)
write_frame(
    output_paths["dqcis_depth_summary.csv"],
    dqcis_depth_summary,
)
write_frame(
    output_paths["view_summary.csv"],
    view_summary,
)
write_frame(
    output_paths["namespace_summary.csv"],
    namespace_summary,
)
write_frame(
    output_paths["run_identity_summary.csv"],
    run_identity_summary,
)
write_frame(
    output_paths["summary.csv"],
    summary,
)
write_frame(
    output_paths["paired_comparisons.csv"],
    paired_comparisons,
)
write_gzip_frame(
    output_paths["metrics_by_query.csv.gz"],
    metrics_by_query.sort_values(
        ["system", "sample_id"],
        kind="mergesort",
    ),
)
write_frame(
    output_paths["qid_mapping.csv"],
    qid_mapping,
)
write_gzip_jsonl(
    output_paths["rankings_top100.jsonl.gz"],
    (
        {
            "system": system,
            "sample_id": sample_id,
            "docids": runs[system][sample_id],
        }
        for system in METHOD_ORDER
        for sample_id in sorted(runs[system])
    ),
)

input_identity = {
    "protocol": PROTOCOL,
    "dataset": "TopiOCQA dev",
    "population_sha256": EXPECTED_POPULATION_SHA256,
    "query_bundle_sha256": EXPECTED_QUERY_BUNDLE_SHA256,
    "ance_rankings_sha256": EXPECTED_ANCE_RANKING_SHA256,
    "dqcis_topiocqa_sha256": DQ_CIS_SHA256["topiocqa"],
    "dqcis_qrecc_showcase_sha256": DQ_CIS_SHA256["qrecc"],
    "nb05_pretrained_sha256": NB05_PRETRAINED_SHA256,
    "budget": BUDGET,
    "top_k": TOP_K,
    "rrf_k": RRF_K,
    "methods": list(METHOD_ORDER),
    "comparisons": [
        list(spec)
        for spec in COMPARISON_SPECS
    ],
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "bootstrap_seed_base": (
        BOOTSTRAP_SEED_BASE
    ),
}
write_json(
    RESULT_DIR / "manifest.json",
    {
        "schema_version": 3,
        "complete": True,
        "selection_performed": False,
        "qrecc_evaluated": False,
        "input_identity": input_identity,
        "files": {
            name: sha256_file(path)
            for name, path in output_paths.items()
        },
    },
)


### Conclusion

ROCC’s value extends beyond improving IterCQR in isolation. The published DQ-CIS ChatGPT/ColBERTv2 ranking already reaches 0.3310 MRR. Fusion with the budgeted baseline $I_{64}$ IterCQR→ANCE ranking reaches 0.3320, while fusion with the ROCC-selected route reaches 0.3461 MRR, 0.5879 $\mathrm{Recall}@10$, and 0.7924 $\mathrm{Recall}@100$.

The controlled `DQ+R − DQ+I` comparison yields a query-level MRR gain of 0.0141 with a 95% confidence interval of `[0.0045, 0.0238]`. History selection is therefore responsible for the improvement. ROCC focuses the added dense ranking on useful conversational evidence, allowing RRF to improve both early relevance and retrieval coverage.

The direct gain over DQ-CIS is 0.0150 MRR with a 95% confidence interval of `[0.0019, 0.0280]`. ROCC provides a complementary retrieval route that improves fusion with a stronger external semantic baseline.

## Result summary

These tables read the saved results of this notebook and display them directly, without a separate export. Rerun the preceding experiment cells first when generating new results.


In [8]:
from experiments.scripts.analysis_results import display_results

display_results(PROJECT_ROOT / "experiments", '08_dqcis_external_ablation')


**MRR**

,system,queries,ranking_depth,MRR,ci95_low,ci95_high
0,DQ,2514,100,0.331036,0.315627,0.346718
1,DQ+I,2514,100,0.332011,0.317038,0.346751
2,DQ+R,2514,100,0.346066,0.331145,0.361022


**paired MRR differences**

,left,right,n_queries,delta_mrr,ci95_low,ci95_high
0,DQ+R,DQ+I,2514,0.014055,0.004453,0.023831
1,DQ+R,DQ,2514,0.015030,0.001943,0.027974
2,DQ+I,DQ,2514,0.000975,-0.011468,0.013771
